# Lambda Preprocessing Pipeline - Data Analysis
**Purpose**: Analyze preprocessing outputs to validate data quality, distribution, and completeness

**Sections**:
1. Setup and Data Loading
2. Overall Pipeline Metrics
3. Text Features Analysis
4. Image Features Analysis
5. Structured Features Analysis
6. Multimodal Completeness
7. Quality Checks and Validation
8. Summary Report

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import json
import torch
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✅ Libraries imported successfully")

In [ ]:
# Configuration
# Lambda GPU instance paths (absolute)
OUTPUT_DIR = Path('/home/ubuntu/mimic-cxr-validation/step2_preprocessing/output/validation_200/train')
COHORT_PATH = Path('/home/ubuntu/mimic-cxr-validation/step2_preprocessing/cohorts/validation_subset_200_with_reports.csv')
STATS_PATH = Path('/home/ubuntu/mimic-cxr-validation/step2_preprocessing/output/validation_200/train/processing_stats.json')

# Verify paths exist
if OUTPUT_DIR.exists():
    print(f"✅ Output directory found: {OUTPUT_DIR}")
    print(f"   - Images: {len(list((OUTPUT_DIR / 'images').glob('*.pt')))} files")
    print(f"   - Text: {len(list((OUTPUT_DIR / 'text_features').glob('*.pt')))} files")
    print(f"   - Structured: {len(list((OUTPUT_DIR / 'structured_features').glob('*.json')))} files")
    print(f"   - Metadata: {len(list((OUTPUT_DIR / 'metadata').glob('*.json')))} files")
else:
    print(f"⚠️  Output directory not found: {OUTPUT_DIR}")
    print("   Please adjust OUTPUT_DIR to point to your preprocessing outputs")

In [ ]:
# Load cohort
if COHORT_PATH.exists():
    cohort = pd.read_csv(COHORT_PATH)
    print(f"✅ Cohort loaded: {len(cohort)} samples")
    print(f"   Columns: {list(cohort.columns[:10])}...")
else:
    print(f"⚠️  Cohort file not found: {COHORT_PATH}")
    cohort = None

In [ ]:
# Load processing statistics
if STATS_PATH.exists():
    with open(STATS_PATH, 'r') as f:
        stats = json.load(f)
    print("✅ Processing statistics loaded")
    print(json.dumps(stats.get('statistics', {}), indent=2))
else:
    print(f"⚠️  Statistics file not found: {STATS_PATH}")
    stats = None

## 2. Overall Pipeline Metrics

In [ ]:
# Collect all sample metadata
metadata_dir = OUTPUT_DIR / 'metadata'
all_metadata = []

for meta_file in sorted(metadata_dir.glob('*.json')):
    with open(meta_file, 'r') as f:
        all_metadata.append(json.load(f))

print(f"Loaded metadata for {len(all_metadata)} samples")

# Extract error information
samples_with_errors = [m for m in all_metadata if len(m.get('errors', [])) > 0]
error_free_samples = [m for m in all_metadata if len(m.get('errors', [])) == 0]

print(f"\n📊 Overall Success Rates:")
print(f"   Total samples: {len(all_metadata)}")
print(f"   Error-free: {len(error_free_samples)} ({len(error_free_samples)/len(all_metadata)*100:.1f}%)")
print(f"   With errors: {len(samples_with_errors)} ({len(samples_with_errors)/len(all_metadata)*100:.1f}%)")

In [ ]:
# Analyze error types
error_types = Counter()
for meta in samples_with_errors:
    for error in meta.get('errors', []):
        # Extract error type from error message
        error_type = error.split(':')[0] if ':' in error else error
        error_types[error_type] += 1

print("\n🔍 Error Type Distribution:")
for error_type, count in error_types.most_common():
    print(f"   {error_type}: {count} occurrences")

# Visualize
if error_types:
    fig, ax = plt.subplots(figsize=(10, 5))
    error_df = pd.DataFrame(error_types.most_common(), columns=['Error Type', 'Count'])
    sns.barplot(data=error_df, x='Count', y='Error Type', ax=ax)
    ax.set_title('Error Type Distribution')
    plt.tight_layout()
    plt.show()

## 3. Text Features Analysis

In [ ]:
# Load all text features
text_dir = OUTPUT_DIR / 'text_features'
text_features = []

for text_file in sorted(text_dir.glob('*.pt')):
    data = torch.load(text_file)
    text_features.append({
        'file': text_file.stem,
        'summary': data.get('summary', ''),
        'num_tokens': data['tokens']['num_tokens'] if 'tokens' in data and 'num_tokens' in data['tokens'] else len(data.get('tokens', {}).get('input_ids', [])),
        'num_entities': data.get('num_entities', 0),
        'entities': data.get('entities', [])
    })

text_df = pd.DataFrame(text_features)
print(f"✅ Loaded {len(text_df)} text features")
print(f"\n📊 Text Statistics:")
print(text_df[['num_tokens', 'num_entities']].describe())

In [ ]:
# Text sequence length distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Token count distribution
axes[0].hist(text_df['num_tokens'], bins=50, edgecolor='black')
axes[0].axvline(text_df['num_tokens'].mean(), color='red', linestyle='--', label=f'Mean: {text_df["num_tokens"].mean():.1f}')
axes[0].axvline(text_df['num_tokens'].median(), color='green', linestyle='--', label=f'Median: {text_df["num_tokens"].median():.1f}')
axes[0].set_xlabel('Number of Tokens')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Text Sequence Length Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Entity count distribution
axes[1].hist(text_df['num_entities'], bins=30, edgecolor='black')
axes[1].axvline(text_df['num_entities'].mean(), color='red', linestyle='--', label=f'Mean: {text_df["num_entities"].mean():.1f}')
axes[1].set_xlabel('Number of Entities')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Named Entity Count Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📈 Text Coverage:")
print(f"   Samples with tokens > 2: {(text_df['num_tokens'] > 2).sum()} ({(text_df['num_tokens'] > 2).sum()/len(text_df)*100:.1f}%)")
print(f"   Samples with entities: {(text_df['num_entities'] > 0).sum()} ({(text_df['num_entities'] > 0).sum()/len(text_df)*100:.1f}%)")
print(f"   Empty summaries: {(text_df['summary'] == '').sum()} ({(text_df['summary'] == '').sum()/len(text_df)*100:.1f}%)")

In [ ]:
# Summary length analysis
text_df['summary_length'] = text_df['summary'].str.len()
text_df['summary_word_count'] = text_df['summary'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Character count
axes[0].hist(text_df['summary_length'], bins=50, edgecolor='black')
axes[0].axvline(text_df['summary_length'].mean(), color='red', linestyle='--', label=f'Mean: {text_df["summary_length"].mean():.0f}')
axes[0].set_xlabel('Summary Length (characters)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Claude Summary Character Count')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Word count
axes[1].hist(text_df['summary_word_count'].dropna(), bins=50, edgecolor='black')
axes[1].axvline(text_df['summary_word_count'].mean(), color='red', linestyle='--', label=f'Mean: {text_df["summary_word_count"].mean():.0f}')
axes[1].set_xlabel('Summary Length (words)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Claude Summary Word Count')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Sample summaries
print("\n📝 Sample Claude Summaries:")
print("=" * 80)
for i, row in text_df[text_df['summary_length'] > 0].sample(min(5, len(text_df))).iterrows():
    print(f"\nSample {row['file']}:")
    print(f"Tokens: {row['num_tokens']}, Entities: {row['num_entities']}")
    print(f"Summary: {row['summary'][:200]}..." if len(row['summary']) > 200 else f"Summary: {row['summary']}")
    print("-" * 80)

## 4. Image Features Analysis

In [ ]:
# Load image features
image_dir = OUTPUT_DIR / 'images'
image_features = []

for image_file in sorted(image_dir.glob('*.pt')):
    try:
        img = torch.load(image_file)
        image_features.append({
            'file': image_file.stem,
            'shape': tuple(img.shape),
            'channels': img.shape[0] if len(img.shape) > 0 else 0,
            'height': img.shape[1] if len(img.shape) > 1 else 0,
            'width': img.shape[2] if len(img.shape) > 2 else 0,
            'size_mb': img.element_size() * img.nelement() / (1024 * 1024)
        })
    except Exception as e:
        print(f"Error loading {image_file}: {e}")

image_df = pd.DataFrame(image_features)
print(f"✅ Loaded {len(image_df)} image features")
print(f"\n📊 Image Statistics:")
print(image_df[['channels', 'height', 'width', 'size_mb']].describe())

In [ ]:
# Image shape distribution
shape_counts = image_df['shape'].value_counts()

print(f"\n🖼️  Image Shape Distribution:")
for shape, count in shape_counts.items():
    print(f"   {shape}: {count} images ({count/len(image_df)*100:.1f}%)")

# Visualize image sizes
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Resolution distribution
axes[0].scatter(image_df['width'], image_df['height'], alpha=0.5)
axes[0].set_xlabel('Width')
axes[0].set_ylabel('Height')
axes[0].set_title('Image Resolution Distribution')
axes[0].grid(True, alpha=0.3)

# File size distribution
axes[1].hist(image_df['size_mb'], bins=30, edgecolor='black')
axes[1].axvline(image_df['size_mb'].mean(), color='red', linestyle='--', label=f'Mean: {image_df["size_mb"].mean():.2f} MB')
axes[1].set_xlabel('Size (MB)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Image File Size Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Structured Features Analysis

In [ ]:
# Load structured features
structured_dir = OUTPUT_DIR / 'structured_features'
structured_features = []

for struct_file in sorted(structured_dir.glob('*.json')):
    with open(struct_file, 'r') as f:
        data = json.load(f)
        structured_features.append({
            'file': struct_file.stem,
            'data': data
        })

print(f"✅ Loaded {len(structured_features)} structured feature files")

In [ ]:
# Analyze DICOM metadata coverage (first 10 fields)
dicom_fields = [
    'view_pa', 'view_ap', 'view_lateral',
    'orientation_erect', 'orientation_recumbent', 'orientation_unknown',
    'is_portable', 'image_rows_normalized', 'image_cols_normalized', 'num_views'
]

dicom_coverage = {field: 0 for field in dicom_fields}
dicom_values = {field: [] for field in dicom_fields}

for item in structured_features:
    data = item['data']
    for field in dicom_fields:
        if field in data and data[field] is not None:
            dicom_coverage[field] += 1
            dicom_values[field].append(data[field])

print("\n🏥 DICOM Metadata Coverage:")
for field, count in dicom_coverage.items():
    pct = count / len(structured_features) * 100 if structured_features else 0
    print(f"   {field}: {count}/{len(structured_features)} ({pct:.1f}%)")

In [ ]:
# Visualize DICOM metadata
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# View position distribution
view_counts = {
    'PA': sum([1 for v in dicom_values['view_pa'] if v == 1.0]),
    'AP': sum([1 for v in dicom_values['view_ap'] if v == 1.0]),
    'LATERAL': sum([1 for v in dicom_values['view_lateral'] if v == 1.0])
}
axes[0, 0].bar(view_counts.keys(), view_counts.values())
axes[0, 0].set_title('View Position Distribution')
axes[0, 0].set_ylabel('Count')
axes[0, 0].grid(True, alpha=0.3)

# Orientation distribution
orientation_counts = {
    'Erect': sum([1 for v in dicom_values['orientation_erect'] if v == 1.0]),
    'Recumbent': sum([1 for v in dicom_values['orientation_recumbent'] if v == 1.0]),
    'Unknown': sum([1 for v in dicom_values['orientation_unknown'] if v == 1.0])
}
axes[0, 1].bar(orientation_counts.keys(), orientation_counts.values())
axes[0, 1].set_title('Patient Orientation Distribution')
axes[0, 1].set_ylabel('Count')
axes[0, 1].grid(True, alpha=0.3)

# Portable detection
portable_counts = {
    'Portable': sum([1 for v in dicom_values['is_portable'] if v == 1.0]),
    'Not Portable': sum([1 for v in dicom_values['is_portable'] if v == 0.0])
}
axes[1, 0].bar(portable_counts.keys(), portable_counts.values())
axes[1, 0].set_title('Portable vs Non-Portable')
axes[1, 0].set_ylabel('Count')
axes[1, 0].grid(True, alpha=0.3)

# Number of views distribution
axes[1, 1].hist([v for v in dicom_values['num_views'] if v is not None], bins=range(1, 12), edgecolor='black')
axes[1, 1].set_title('Number of Views per Study')
axes[1, 1].set_xlabel('Number of Views')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze vital signs coverage
vital_fields = [
    'vital_temperature', 'vital_heart_rate', 'vital_resp_rate',
    'vital_o2_sat', 'vital_sbp', 'vital_dbp'
]

vital_coverage = {field: {'present': 0, 'missing': 0, 'values': []} for field in vital_fields}

for item in structured_features:
    data = item['data']
    for field in vital_fields:
        if field in data:
            if isinstance(data[field], dict):
                if data[field].get('is_missing', True) or data[field].get('value') == 'NOT_DONE':
                    vital_coverage[field]['missing'] += 1
                else:
                    vital_coverage[field]['present'] += 1
                    try:
                        vital_coverage[field]['values'].append(float(data[field]['value']))
                    except:
                        pass

print("\n❤️  Vital Signs Coverage:")
for field, counts in vital_coverage.items():
    total = counts['present'] + counts['missing']
    pct = counts['present'] / total * 100 if total > 0 else 0
    print(f"   {field}: {counts['present']}/{total} ({pct:.1f}%)")
    if counts['values']:
        mean_val = np.mean(counts['values'])
        print(f"      Mean value: {mean_val:.1f}")

In [ ]:
# Visualize vital signs distribution
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

vital_labels = {
    'vital_temperature': 'Temperature (°C)',
    'vital_heart_rate': 'Heart Rate (bpm)',
    'vital_resp_rate': 'Respiratory Rate',
    'vital_o2_sat': 'O2 Saturation (%)',
    'vital_sbp': 'Systolic BP (mmHg)',
    'vital_dbp': 'Diastolic BP (mmHg)'
}

for idx, (field, label) in enumerate(vital_labels.items()):
    values = vital_coverage[field]['values']
    if values:
        axes[idx].hist(values, bins=30, edgecolor='black', alpha=0.7)
        axes[idx].axvline(np.mean(values), color='red', linestyle='--', 
                         label=f'Mean: {np.mean(values):.1f}')
        axes[idx].set_title(label)
        axes[idx].set_xlabel('Value')
        axes[idx].set_ylabel('Frequency')
        axes[idx].legend()
        axes[idx].grid(True, alpha=0.3)
    else:
        axes[idx].text(0.5, 0.5, 'No Data', ha='center', va='center', fontsize=16)
        axes[idx].set_title(label)

plt.tight_layout()
plt.show()

## 6. Multimodal Completeness Analysis

In [ ]:
# Analyze multimodal completeness
completeness = []

for meta in all_metadata:
    sample_id = f"s{meta['subject_id']}_study{meta['study_id']}"
    
    # Check if each modality exists
    has_image = (OUTPUT_DIR / 'images' / f"{sample_id}.pt").exists()
    has_text = (OUTPUT_DIR / 'text_features' / f"{sample_id}.pt").exists()
    has_structured = (OUTPUT_DIR / 'structured_features' / f"{sample_id}.json").exists()
    
    # Check if text is non-empty (more than 2 tokens)
    text_valid = False
    if has_text:
        text_data = torch.load(OUTPUT_DIR / 'text_features' / f"{sample_id}.pt")
        num_tokens = text_data['tokens']['num_tokens'] if 'tokens' in text_data and 'num_tokens' in text_data['tokens'] else len(text_data.get('tokens', {}).get('input_ids', []))
        text_valid = num_tokens > 2
    
    completeness.append({
        'sample_id': sample_id,
        'has_image': has_image,
        'has_text': has_text,
        'text_valid': text_valid,
        'has_structured': has_structured,
        'modality_count': sum([has_image, text_valid, has_structured]),
        'all_complete': has_image and text_valid and has_structured
    })

comp_df = pd.DataFrame(completeness)

print("\n🎯 Multimodal Completeness:")
print(f"   Total samples: {len(comp_df)}")
print(f"   Image modality: {comp_df['has_image'].sum()} ({comp_df['has_image'].sum()/len(comp_df)*100:.1f}%)")
print(f"   Text modality (valid): {comp_df['text_valid'].sum()} ({comp_df['text_valid'].sum()/len(comp_df)*100:.1f}%)")
print(f"   Structured modality: {comp_df['has_structured'].sum()} ({comp_df['has_structured'].sum()/len(comp_df)*100:.1f}%)")
print(f"\n   Complete samples (all 3 modalities): {comp_df['all_complete'].sum()} ({comp_df['all_complete'].sum()/len(comp_df)*100:.1f}%)")

# Modality count distribution
modality_counts = comp_df['modality_count'].value_counts().sort_index()
print("\n   Samples by modality count:")
for count, num_samples in modality_counts.items():
    print(f"      {count} modalities: {num_samples} samples ({num_samples/len(comp_df)*100:.1f}%)")

In [ ]:
# Visualize multimodal completeness
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Modality presence
modality_presence = {
    'Image': comp_df['has_image'].sum(),
    'Text (Valid)': comp_df['text_valid'].sum(),
    'Structured': comp_df['has_structured'].sum()
}
axes[0].bar(modality_presence.keys(), modality_presence.values())
axes[0].axhline(len(comp_df), color='red', linestyle='--', label=f'Total samples: {len(comp_df)}')
axes[0].set_title('Modality Presence')
axes[0].set_ylabel('Number of Samples')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Completeness distribution
axes[1].bar(modality_counts.index, modality_counts.values, edgecolor='black')
axes[1].set_title('Samples by Number of Modalities')
axes[1].set_xlabel('Number of Modalities Present')
axes[1].set_ylabel('Number of Samples')
axes[1].set_xticks([0, 1, 2, 3])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Quality Checks and Validation

In [ ]:
# Identify potential issues
print("🔍 Quality Checks:\n")

# 1. Empty text features
empty_text = text_df[text_df['num_tokens'] <= 2]
print(f"1. Empty text features (≤2 tokens): {len(empty_text)} samples")
if len(empty_text) > 0:
    print(f"   Sample IDs: {empty_text['file'].head(5).tolist()}")

# 2. Missing summaries
no_summary = text_df[text_df['summary'].str.len() == 0]
print(f"\n2. Missing Claude summaries: {len(no_summary)} samples")

# 3. Unusual image shapes
shape_counts = image_df['shape'].value_counts()
if len(shape_counts) > 1:
    print(f"\n3. Multiple image shapes detected:")
    for shape, count in shape_counts.items():
        print(f"   {shape}: {count} images")

# 4. Samples with no vitals
no_vitals = 0
for item in structured_features:
    data = item['data']
    has_any_vital = any(
        field in data and 
        isinstance(data[field], dict) and 
        not data[field].get('is_missing', True) and 
        data[field].get('value') != 'NOT_DONE'
        for field in vital_fields
    )
    if not has_any_vital:
        no_vitals += 1

print(f"\n4. Samples with no vital signs: {no_vitals} ({no_vitals/len(structured_features)*100:.1f}%)")

# 5. Incomplete samples
incomplete = comp_df[~comp_df['all_complete']]
print(f"\n5. Incomplete samples (missing modalities): {len(incomplete)} ({len(incomplete)/len(comp_df)*100:.1f}%)")

In [ ]:
# Correlation analysis (if cohort loaded)
if cohort is not None:
    # Merge completeness with cohort demographics
    cohort['sample_id'] = 's' + cohort['subject_id'].astype(str) + '_study' + cohort['study_id'].astype(str)
    merged = cohort.merge(comp_df, on='sample_id', how='left')
    
    print("\n📊 Completeness by Demographics:\n")
    
    if 'gender' in merged.columns:
        by_gender = merged.groupby('gender')['all_complete'].agg(['sum', 'count', 'mean'])
        print("By Gender:")
        print(by_gender)
    
    if 'anchor_age' in merged.columns:
        # Age groups
        merged['age_group'] = pd.cut(merged['anchor_age'], bins=[0, 30, 45, 60, 75, 100], 
                                     labels=['18-30', '31-45', '46-60', '61-75', '76+'])
        by_age = merged.groupby('age_group')['all_complete'].agg(['sum', 'count', 'mean'])
        print("\nBy Age Group:")
        print(by_age)

## 8. Summary Report

In [ ]:
# Generate comprehensive summary report
report = {
    'timestamp': datetime.now().isoformat(),
    'total_samples': len(all_metadata),
    'overall_metrics': {
        'error_free_samples': len(error_free_samples),
        'error_free_rate': len(error_free_samples) / len(all_metadata) if all_metadata else 0,
        'complete_multimodal_samples': comp_df['all_complete'].sum(),
        'complete_multimodal_rate': comp_df['all_complete'].sum() / len(comp_df) if len(comp_df) > 0 else 0
    },
    'text_metrics': {
        'total': len(text_df),
        'valid_text_rate': (text_df['num_tokens'] > 2).sum() / len(text_df) if len(text_df) > 0 else 0,
        'mean_tokens': float(text_df['num_tokens'].mean()),
        'median_tokens': float(text_df['num_tokens'].median()),
        'mean_summary_length': float(text_df['summary_length'].mean()),
        'mean_entities': float(text_df['num_entities'].mean())
    },
    'image_metrics': {
        'total': len(image_df),
        'unique_shapes': len(image_df['shape'].unique()),
        'mean_size_mb': float(image_df['size_mb'].mean()),
        'most_common_shape': str(image_df['shape'].mode()[0]) if len(image_df) > 0 else None
    },
    'structured_metrics': {
        'total': len(structured_features),
        'dicom_coverage': {
            field: dicom_coverage[field] / len(structured_features) if structured_features else 0
            for field in dicom_fields
        },
        'vital_coverage': {
            field: vital_coverage[field]['present'] / (vital_coverage[field]['present'] + vital_coverage[field]['missing']) 
            if (vital_coverage[field]['present'] + vital_coverage[field]['missing']) > 0 else 0
            for field in vital_fields
        }
    },
    'quality_flags': {
        'empty_text_count': int((text_df['num_tokens'] <= 2).sum()),
        'no_summary_count': int((text_df['summary'].str.len() == 0).sum()),
        'no_vitals_count': no_vitals,
        'incomplete_samples': int((~comp_df['all_complete']).sum())
    }
}

# Save report
report_path = OUTPUT_DIR.parent / 'pipeline_analysis_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print("\n" + "="*80)
print("📋 PREPROCESSING PIPELINE ANALYSIS SUMMARY")
print("="*80)
print(f"\nTotal Samples: {report['total_samples']}")
print(f"\nOverall Success Rates:")
print(f"  - Error-free: {report['overall_metrics']['error_free_rate']:.1%}")
print(f"  - Complete multimodal: {report['overall_metrics']['complete_multimodal_rate']:.1%}")

print(f"\nText Features:")
print(f"  - Valid text rate: {report['text_metrics']['valid_text_rate']:.1%}")
print(f"  - Mean tokens: {report['text_metrics']['mean_tokens']:.1f}")
print(f"  - Mean summary length: {report['text_metrics']['mean_summary_length']:.0f} characters")

print(f"\nImage Features:")
print(f"  - Total images: {report['image_metrics']['total']}")
print(f"  - Mean size: {report['image_metrics']['mean_size_mb']:.2f} MB")

print(f"\nStructured Features:")
print(f"  - DICOM metadata coverage: {np.mean(list(report['structured_metrics']['dicom_coverage'].values())):.1%}")
print(f"  - Vital signs coverage: {np.mean(list(report['structured_metrics']['vital_coverage'].values())):.1%}")

print(f"\nQuality Flags:")
print(f"  - Empty text: {report['quality_flags']['empty_text_count']} samples")
print(f"  - No vitals: {report['quality_flags']['no_vitals_count']} samples")
print(f"  - Incomplete: {report['quality_flags']['incomplete_samples']} samples")

print(f"\n✅ Report saved to: {report_path}")
print("="*80)

In [ ]:
# Final visualization: Summary dashboard
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# Overall success rates
ax1 = fig.add_subplot(gs[0, :])
metrics = ['Error-free', 'Complete\nMultimodal', 'Valid Text', 'Has Images', 'Has Structured']
values = [
    report['overall_metrics']['error_free_rate'],
    report['overall_metrics']['complete_multimodal_rate'],
    report['text_metrics']['valid_text_rate'],
    len(image_df) / len(all_metadata) if all_metadata else 0,
    len(structured_features) / len(all_metadata) if all_metadata else 0
]
colors = ['green' if v >= 0.95 else 'orange' if v >= 0.8 else 'red' for v in values]
ax1.barh(metrics, values, color=colors, alpha=0.7, edgecolor='black')
ax1.set_xlim(0, 1)
ax1.set_xlabel('Success Rate')
ax1.set_title('Overall Pipeline Success Metrics', fontsize=14, fontweight='bold')
ax1.axvline(0.95, color='green', linestyle='--', alpha=0.5, label='95% target')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Text tokens
ax2 = fig.add_subplot(gs[1, 0])
ax2.hist(text_df['num_tokens'], bins=30, edgecolor='black', alpha=0.7)
ax2.set_title('Text Sequence Length')
ax2.set_xlabel('Tokens')
ax2.grid(True, alpha=0.3)

# DICOM coverage
ax3 = fig.add_subplot(gs[1, 1])
dicom_cov_values = list(report['structured_metrics']['dicom_coverage'].values())
ax3.bar(range(len(dicom_cov_values)), dicom_cov_values, alpha=0.7, edgecolor='black')
ax3.set_title('DICOM Metadata Coverage')
ax3.set_ylabel('Coverage Rate')
ax3.set_ylim(0, 1)
ax3.grid(True, alpha=0.3)

# Vital signs coverage
ax4 = fig.add_subplot(gs[1, 2])
vital_cov_values = list(report['structured_metrics']['vital_coverage'].values())
ax4.bar(range(len(vital_cov_values)), vital_cov_values, alpha=0.7, edgecolor='black', color='coral')
ax4.set_title('Vital Signs Coverage')
ax4.set_ylabel('Coverage Rate')
ax4.set_ylim(0, 1)
ax4.grid(True, alpha=0.3)

# Modality completeness
ax5 = fig.add_subplot(gs[2, 0])
ax5.bar(modality_counts.index, modality_counts.values, alpha=0.7, edgecolor='black', color='steelblue')
ax5.set_title('Modality Completeness')
ax5.set_xlabel('Number of Modalities')
ax5.set_ylabel('Sample Count')
ax5.grid(True, alpha=0.3)

# Error types
ax6 = fig.add_subplot(gs[2, 1:])
if error_types:
    error_items = list(error_types.most_common(10))
    ax6.barh([e[0][:30] for e in error_items], [e[1] for e in error_items], alpha=0.7, edgecolor='black', color='salmon')
    ax6.set_title('Top 10 Error Types')
    ax6.set_xlabel('Count')
    ax6.grid(True, alpha=0.3)
else:
    ax6.text(0.5, 0.5, 'No Errors!', ha='center', va='center', fontsize=20, color='green')
    ax6.set_title('Error Analysis')

plt.suptitle('Preprocessing Pipeline Analysis Dashboard', fontsize=16, fontweight='bold', y=0.995)
plt.savefig(OUTPUT_DIR.parent / 'pipeline_analysis_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Dashboard saved to: {OUTPUT_DIR.parent / 'pipeline_analysis_dashboard.png'}")